In [ ]:
import pandas as pd
import numpy as np

# Fuente original del dataset
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

# 1) Se descartan las poquísimas filas sin Android Ver / Type / Content Rating (MCAR, impacto nulo)
df = df.dropna(subset=["Android Ver", "Type", "Content Rating"])

# 2) El Rating faltante se imputa con la mediana de su propia categoría (no la global)
df["Rating"] = df["Rating"].fillna(
    df.groupby("Category")["Rating"].transform("median")
)

print("Registros en el archivo limpio:", len(df))
df.head()

Registros en el archivo limpio: 10837


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Reto 1: La Trampa de las Unidades de Medida

La columna `Size` mezcla unidades (`"19M"`, `"201k"`) y el texto `"Varies with device"`.
Escribimos una **función pura** que normaliza todo a **Megabytes**:

- termina en `M` → se quita la `M` y se convierte a `float` (ya está en MB).
- termina en `k` → se quita la `k`, se convierte a `float` y se divide entre `1024` (kB → MB).
- `"Varies with device"` → `np.nan`.

La función no muta nada externo: para la misma entrada siempre devuelve la misma salida.

In [ ]:
def size_a_megabytes(valor):
    if not isinstance(valor, str):
        return np.nan

    texto = valor.strip()

    if texto == "Varies with device":
        return np.nan
    if texto.endswith("M"):
        return float(texto[:-1])
    if texto.endswith("k"):
        return float(texto[:-1]) / 1024

    return np.nan                            # cualquier otro formato inesperado


df["Size"] = df["Size"].apply(size_a_megabytes)
df["Size"].head(10)

0    19.0
1    14.0
2     8.7
3    25.0
4     2.8
5     5.6
6    19.0
7    29.0
8    33.0
9     3.1
Name: Size, dtype: float64

In [ ]:
peso_promedio_mb = df["Size"].mean()
print(f"Peso promedio de las apps: {peso_promedio_mb:.2f} MB")

Peso promedio de las apps: 21.52 MB



_El **peso promedio** de las aplicaciones de la Play Store es de **≈ 21.52 MB**
(.mean() ignora automáticamente los NaN de las apps "Varies with device").
Ese valor da al equipo de infraestructura una línea base para estimar consumo de almacenamiento
y ancho de banda de descarga por app_


## Reto 2: El Tipo de Dato Cronológico


In [ ]:
# Sobrescribimos la columna: de object (string) a datetime64
df["Last Updated"] = pd.to_datetime(df["Last Updated"], format="%B %d, %Y")

df["Year_Updated"] = df["Last Updated"].dt.year

print(df["Last Updated"].dtype)
df[["Last Updated", "Year_Updated"]].head()

datetime64[us]


,Last Updated,Year_Updated
0,2018-01-07,2018
1,2018-01-15,2018
2,2018-08-01,2018
3,2018-06-08,2018
4,2018-06-20,2018


In [ ]:
df["Year_Updated"].value_counts()

Year_Updated
2018    7346
2017    1867
2016     804
2015     459
2014     209
2013     110
2012      26
2011      15
2010       1
Name: count, dtype: int64


_El año con mayor cantidad de aplicaciones actualizadas es **2018**, con **7,346** apps
(muy por encima de 2017 con 1,867). El dataset es una fotografía tomada a mediados de 2018,
así que la mayoría de las apps activas tenían una actualización reciente de ese mismo año_


## Reto 3: La Decisión Arquitectónica


In [ ]:
nulos_size = df["Size"].isna().sum()
total = len(df)
print(f"Registros con Size nulo: {nulos_size} de {total} ({nulos_size / total:.1%})")
print(f"Mediana global del peso: {df['Size'].median():.2f} MB")

Registros con Size nulo: 1694 de 10837 (15.6%)
Mediana global del peso: 13.00 MB


In [ ]:
# Decisión aplicada: eliminar las filas sin peso real (dropna sobre Size).
df_modelo = df.dropna(subset=["Size"]).copy()

print("Antes:", len(df), " | Después de dropna(Size):", len(df_modelo))
print(f"Peso promedio tras dropna: {df_modelo['Size'].mean():.2f} MB")

Antes: 10837  | Después de dropna(Size): 9143
Peso promedio tras dropna: 21.52 MB


_1. Elegimos **.dropna() sobre Size**: el análisis pedido (¿pesan más las apps con menos
   descargas?) tiene el peso como variable **crítica**, y solo perdemos ≈ 15.6 % de las filas,
   una pérdida asumible que deja 9,143 registros_

_2. Imputar la **mediana global (13 MB)** en 1,694 filas colapsaría ese 15 % de los datos en un
   único punto, reduciendo artificialmente la varianza y **aplanando la correlación** entre
   tamaño y descargas que justamente queremos medir_

_3. Además, "Varies with device" no es un dato perdido al azar, sino apps modulares
   (normalmente grandes); asignarles la mediana introduciría un **sesgo sistemático**, mientras
   que descartarlas mantiene limpia la señal para el modelo de predicción_